<center> <img src = https://raw.githubusercontent.com/AndreyRysistov/DatasetsForPandas/main/hh%20label.jpg alt="drawing" style="width:400px;">

**Импорт необходимых модулей.**
**Настройка несколько более удобной, корректной печати DataFrame.**

In [1]:
import pandas as pd
import psycopg2
import requests
from bs4 import BeautifulSoup
from io import StringIO

# рекомендации проверяющих касаемо использования именно функции 'display', не 'print'
from IPython.display import Markdown, display

# реализация согласно заданию, игнорируем предупреждения
import warnings
from requests.exceptions import RequestsDependencyWarning
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy connectable")
warnings.simplefilter("ignore", RequestsDependencyWarning)


pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

import builtins
from tabulate import tabulate

# 1) Сохраняем оригинальный print только один раз
if not hasattr(builtins, "_smartprint_original"):
    builtins._smartprint_original = builtins.print

def _tabulate_df(df: pd.DataFrame, tablefmt="psql", showindex=False):
    return tabulate(df, headers="keys", tablefmt=tablefmt, showindex=showindex)

def _tabulate_series(s: pd.Series, tablefmt="psql", showindex=True, name_col="value"):
    df = s.to_frame(name=s.name or name_col)
    return tabulate(df, headers="keys", tablefmt=tablefmt, showindex=showindex)

def smart_print(*args, **kwargs):
    tablefmt = kwargs.pop("tablefmt", "psql")
    showindex = kwargs.pop("showindex", False)
    series_showindex = kwargs.pop("series_showindex", True)

    new_args = []
    for a in args:
        if isinstance(a, pd.DataFrame):
            new_args.append(_tabulate_df(a, tablefmt=tablefmt, showindex=showindex))
        elif isinstance(a, pd.Series):
            new_args.append(_tabulate_series(a, tablefmt=tablefmt, showindex=series_showindex))
        else:
            new_args.append(a)

    # 2) Всегда печатаем через сохранённый оригинал
    return builtins._smartprint_original(*new_args, **kwargs)

# 3) Подменяем print
builtins.print = smart_print



**Подключение к БД**

In [ ]:
# в соответствии с указанием по модулю, данные подключения удалены
DBNAME = ''
USER = ''
PASSWORD = ''
HOST = ''
PORT = ''

connection = psycopg2.connect(
   dbname=DBNAME,
   user=USER,
   host=HOST,
   password=PASSWORD,
   port=PORT
)

<br><br><br><br>
# Предварительный анализ данных

<br/><br/>
**Задание № 1**

**Количество вакансий (таблица vacancies).**

In [3]:
request = f'''
select 
    count(id) as vac_count
from vacancies
'''

vac_count = pd.read_sql_query(request, connection)
display(Markdown(f'Количество вакансий в БД: **{vac_count['vac_count'].iloc[0]}**'))

Количество вакансий в БД: **49197**

<br/><br/>
**Задание № 2**

**Количество работодателей (таблица employers).**

In [4]:
request = f'''
select 
    count(id) as emp_count
from employers
'''

emp_count = pd.read_sql_query(request, connection)
display(Markdown(f'Количество работодателей в БД: **{emp_count['emp_count'].iloc[0]}**'))

Количество работодателей в БД: **23501**

<br/><br/>
**Задание № 3**

**Количество регионов (таблица areas).**

In [5]:
request = f'''
select 
    count(id) as area_count
from areas
'''

area_count = pd.read_sql_query(request, connection)
display(Markdown(f'Количество регионов в БД: **{area_count['area_count'].iloc[0]}**'))

Количество регионов в БД: **1362**

<br/><br/>
**Задание № 4**

**Количество сфер деятельности (таблица industries).**

In [6]:
request = f'''
select 
    count(id) as ind_count
from industries
'''

ind_count = pd.read_sql_query(request, connection)
display(Markdown(f'Количество сфер деятельности в БД: **{ind_count['ind_count'].iloc[0]}**'))

Количество сфер деятельности в БД: **294**

<h2 style="color:red;">Итоги по блоку предварительного анализа данных.</h2>

Предварительный анализ указывает на то, что база является репрезентативной, учитывая количество вакансий, работодателей, регионов и сфер деятельности.</br>
В базе представлены как крупные работодатели с большим числом публикаций, так и множейство компаний с единичными или малым количеством вакансий, т.е. идет отражение не только лидеров, но и достаточно большой слой менее заметных участников.</br>
Также данные отражают широкий географический охват, что позвояляет анализировать рынок не только в столицах, но и в разрезе регионов.</br>
Большое число сфер деятельности указывает на то, что вакансии относятся к разным отраслям экономики, что позволяет сопоставлять отрасли между собой.

Общий вывод по предварительному анализу говорит о том, что база вакансий является масштабной, многослойной, географически резделенной, с множеством отраслей экономики, что дает основание говорить о последующих выводах, как о выводах, основанных на достаточно широкой эмпирической базе.

<br><br><br><br>
# Детальный анализ вакансий

<br/><br/>
**Задание № 1**

**Количество вакансий в каждом регионе (cnt, area).**

**Сортировка: количество вакансий, в порядке убывания.**

In [7]:
request = f'''
select
    count(vacancies.id) as cnt
    ,areas.name as area
from vacancies
inner join areas
    on vacancies.area_id = areas.id
group by
    areas.id
order by
    cnt desc
'''

task_1 = pd.read_sql_query(request, connection)
display(task_1.head(5))

,cnt,area
0,5333,Москва
1,2851,Санкт-Петербург
2,2112,Минск
3,2006,Новосибирск
4,1892,Алматы


<br/><br/>
**Задание № 2**

**Количество вакансий хотя бы с одним заполненным полем по заработной плате.**

In [8]:
request = f'''
select
    count(vacancies.id) as not_empty_vac_count
from vacancies
where
    salary_from is not null or salary_to is not null
'''

task_2 = pd.read_sql_query(request, connection)
display(Markdown(f'Количество вакансий хотя бы с одним заполненным полем по ЗП: **{task_2['not_empty_vac_count'].iloc[0]}**'))

Количество вакансий хотя бы с одним заполненным полем по ЗП: **24073**

<br/><br/>
**Задание № 3**

**Среднее значений нижней и верхней зарплатной вилки (округленное до целого).**

In [9]:
request = f'''
select
    avg(vacancies.salary_from) as salary_from_avg
    ,avg(vacancies.salary_to) as salary_to_avg
from vacancies
'''

task_3 = pd.read_sql_query(request, connection)
display(Markdown(
    f"Среднее значение нижней зарплатной вилки: **{round(task_3['salary_from_avg'].iloc[0])}**  \n"
    f"Среднее значение верхней зарплатной вилки: **{round(task_3['salary_to_avg'].iloc[0])}**"
))

Среднее значение нижней зарплатной вилки: **71065**  
Среднее значение верхней зарплатной вилки: **110537**

<br/><br/>
**Задание № 4**

**Количество вакансий для пар 'Тип рабочего графика' (schedule) - 'Тип трудоустройства' (employment).**

**Сортировка: количество вакансий, по убыванию.**

In [10]:
request = f'''
select
    schedule
    ,employment
    ,count(vacancies.id) as vac_count
from vacancies
group by
    schedule
    ,employment
order by
    vac_count desc
'''

task_4 = pd.read_sql_query(request, connection)
display(task_4.head(5))

,schedule,employment,vac_count
0,Полный день,Полная занятость,35367
1,Удаленная работа,Полная занятость,7802
2,Гибкий график,Полная занятость,1593
3,Удаленная работа,Частичная занятость,1312
4,Сменный график,Полная занятость,940


<br/><br/>
**Задание № 5**

**Значения поля 'Требуемый опыт работы' (experience).**

**Сортировка: количество упоминаний указанного значения среди вакансий, по возрастанию.**

In [11]:
request = f'''
select
    experience
from vacancies
group by
    experience
order by
    count(vacancies.id) asc
'''

task_5 = pd.read_sql_query(request, connection)
display(task_5)

,experience
0,Более 6 лет
1,Нет опыта
2,От 3 до 6 лет
3,От 1 года до 3 лет


<h2 style="color:red;">Итоги по блоку детального анализа вакансий.</h2>

Детальный анализ вакансий показывает, что рынок труда распределен крайне неравномерно. Наибольшее число вакансий сосредоточено в крупных городах, таких как Москва, Санк-Петербург, Минск, Новосибирск, Алматы, что позволяет сделать вывод о высокой концентрации спроса в крупных деловых и технологических центрах.</br>
Лишь половина вакансий содеражат данные о заработной плате, что соответственно дает нам право говорить о том, что работодатели не склонны раскрывать информацию о заработной плате на этапе публикации вакансии.</br>
Заметный разрыв между нижней и верхней границами зарплатной вилки указывает на высокую вариативность оплаты труда, что вероятно связано с нюансами опыта работы, специализации, регионом, отраслью и конкретной компанией. Тажке, вероятно, компании оставляют пространство для переговоров.</br>
Прослеживается явное превалирование стандартной модели рабочего графика, т.е. полный рабочий день, полная занятость. Затем идет удаленная работа с полной занятостью, что говорит о том, что традиционный формат рабоыт по прежнему остается основным на рынке труда, но вариант удаленный работы тоже занял свою устойчивую позицию.</br>
Выборка по опыту работы дает понять, что спрос ориентирован прежде всего на кандидатов со сформированным базовым опытом, а не на новичков или сотрудников с высоким опытом работы.</br>

Общий вывод по анализу вакансий говорит о том, что рынок сосредоточен в крупных городах, в основном ориентирован на полную занятость, не особо прозрачен по части заработной платы и ориентирован на специалистов с небольшим, но устойчивым опытом.

<br><br><br><br>
# Анализ работодателей

<br/><br/>
**Задание № 1**

**1-ое и 5-ое место среди работодателей по количеству вакансий.**

In [12]:
request = f'''
with draft_info as
(select
    -- row_number используем для нумерации
    row_number() over(order by count(vacancies.id) desc) as place
    ,employers.name
from employers
left join vacancies
    on employers.id = vacancies.employer_id
group by
    employers.id
order by place asc)

select *
from draft_info
where place in (1,5)
'''

task_1 = pd.read_sql_query(request, connection)
display(Markdown(
    f"**Первое** место среди работодателей по количеству вакансий: **{task_1['name'].iloc[0]}**  \n"
    f"**Пятое** место среди работодателей по количеству вакансий: **{task_1['name'].iloc[1]}**"
))

**Первое** место среди работодателей по количеству вакансий: **Яндекс**  
**Пятое** место среди работодателей по количеству вакансий: **Газпром нефть**

<br/><br/>
**Задание № 2**

**Регион, количество работодателей в регионе, количество вакансий в регионе.**

**Найти регион с отсутствующими вакансиями и наибольшим количеством работодателей.**

In [13]:
request = f'''
select
    areas.name
    ,coalesce(sub_emp.emp_count, 0) as emp_count
    ,coalesce(sub_vac.vac_count, 0) as vac_count
from areas
-- sub_emp реализуем ввиде подзапроса и left join к нему
left join
    (select
        area
        ,count(id) as emp_count
    from employers
    group by
        area) sub_emp on areas.id = sub_emp.area
-- sub_vac реализуем ввиде подзапроса и left join к нему
left join
    (select
        area_id
        ,count(id) as vac_count
    from vacancies
    group by
        area_id) sub_vac on areas.id = sub_vac.area_id
'''

task_2 = pd.read_sql_query(request, connection)
# общий вид запроса
display(task_2.head(5))

# регион с отсутствующими вакансиями и наибольшим количеством работодателей
task_2_ask_result = task_2[task_2['vac_count']==0].sort_values(by='emp_count', ascending=False)['name'].head(1).iloc[0]
display(Markdown(f'Регион с отсутствующими вакансиями и наибольшим количеством работодателей: **{task_2_ask_result}**'))

,name,emp_count,vac_count
0,Тбилиси,13,474
1,Майкоп,7,21
2,Нерюнгри,0,6
3,Новокузнецк,71,156
4,Санкт-Петербург,2217,2851


Регион с отсутствующими вакансиями и наибольшим количеством работодателей: **Россия**

<br/><br/>
**Задание № 3**

**Работодатели и число регионов, в которых они публикуют свои вакансии.**

**Сортировка: количество регионов, по убыванию.**

In [14]:
request = f'''
select
    employers.name
    ,count(distinct vacancies.area_id) as area_count
from employers
left join vacancies
    on employers.id = vacancies.employer_id
group by
    employers.id
order by area_count desc
'''

task_3 = pd.read_sql_query(request, connection)
# общий вид запроса
print(task_3.head(5))

# максимальное значение
display(Markdown(f'Максимальное значение: **{task_3['area_count'].iloc[0]}**'))

+------------------------+--------------+
| name                   |   area_count |
|------------------------+--------------|
| Яндекс                 |          181 |
| Ростелеком             |          152 |
| Спецремонт             |          116 |
| Поляков Денис Иванович |           88 |
| ООО ЕФИН               |           71 |
+------------------------+--------------+


Максимальное значение: **181**

<br/><br/>
**Задание № 4**

**Количество работодателей с отсутствующей сферой деятельности.**

In [15]:
request = f'''
select
    count(employers.id) as emp_empty_ind_count
from employers
left join employers_industries
    on employers.id = employers_industries.employer_id
where
    employers_industries.industry_id is null
'''

task_4 = pd.read_sql_query(request, connection)
# количество работодателей с отсутствующей сферой деятельности
display(Markdown(f'Количество работодателей с отсутствующей сферой деятельности: **{task_4['emp_empty_ind_count'].iloc[0]}**'))

Количество работодателей с отсутствующей сферой деятельности: **8419**

<br/><br/>
**Задание № 5**

**Наименование компании, у которой указано четыре сферы деятельности, и которая находится на третьем месте в алфавитном порядке.**

In [16]:
request = f'''
select
    employers.name
from employers
inner join employers_industries
    on employers.id = employers_industries.employer_id
group by
    employers.name
having
    count(employers.id) = 4
order by
    employers.name asc
offset 2
limit 1
'''

task_5 = pd.read_sql_query(request, connection)
# наименование компании, у которой указано четыре сферы деятельности, и которая находится на третьем месте в алфавитном порядке
display(Markdown(f'Наименование компании, у которой указано четыре сферы деятельности, и которая находится на третьем месте в алфавитном порядке: **{task_5['name'].iloc[0]}**'))

Наименование компании, у которой указано четыре сферы деятельности, и которая находится на третьем месте в алфавитном порядке: **2ГИС**

<br/><br/>
**Задание № 6**

**Количество работодателей, у которых в качестве сферы деятельности указано - 'Разработка программного обеспечения'.**

In [17]:
request = f'''
select
    count(distinct employers.id) as task_6_emp_count
from employers
inner join employers_industries
    on employers.id = employers_industries.employer_id
inner join industries
    on employers_industries.industry_id = industries.id
    and industries.name = 'Разработка программного обеспечения'
'''

task_6 = pd.read_sql_query(request, connection)
# количество работодателей, у которых в качестве сферы деятельности указано - 'Разработка программного обеспечения'
display(Markdown(f"Количество работодателей, у которых в качестве сферы деятельности указано - 'Разработка программного обеспечения': **{task_6['task_6_emp_count'].iloc[0]}**"))

Количество работодателей, у которых в качестве сферы деятельности указано - 'Разработка программного обеспечения': **3553**

<br/><br/>
**Задание № 7**

**Для компании 'Яндекс' отобразить список городов - милионников, в которых представлены вакансии. Также отобразить количество вакансий. Также добавить строку 'Total' с общим количеством вакансий компании.**

**Сортировка: количество вакансий, по возрастанию.**

In [18]:
url = "https://ru.wikipedia.org/wiki/Города-миллионники_России"

# перестраховка на случай блокироки на парсинг со стороны Википедии
fallback_cities = [
    'Москва', 'Санкт-Петербург', 'Новосибирск', 'Екатеринбург', 'Казань',
    'Красноярск', 'Нижний Новгород', 'Челябинск', 'Уфа', 'Краснодар',
    'Самара', 'Ростов-на-Дону', 'Омск', 'Воронеж', 'Пермь', 'Волгоград'
]

try:
    html = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=10
    ).text
    cities_df = pd.read_html(StringIO(html))[0]
    million_cities = cities_df['Город'].tolist()
except Exception:
    million_cities = fallback_cities

# из списка делаем единую строку с разделителем "' ,'" и обрамляем ее в начальные и конечные одинарные ковычки для последующего использования в условии в запросе
million_cities = "', '".join(million_cities)

request = f"""
select
    areas.name,
    count(vacancies.id) as vac_count
from employers
inner join vacancies
    on employers.id = vacancies.employer_id
inner join areas
    on vacancies.area_id = areas.id
where employers.name = 'Яндекс'
    and areas.name in ('{million_cities}')
group by areas.id, areas.name

union all

select
    'Total' as name,
    count(vacancies.id) as vac_count
from employers
inner join vacancies
    on employers.id = vacancies.employer_id
inner join areas
    on vacancies.area_id = areas.id
where employers.name = 'Яндекс'
    and areas.name in ('{million_cities}')

order by vac_count asc
"""

task_7 = pd.read_sql_query(request, connection)
display(task_7)

,name,vac_count
0,Омск,21
1,Челябинск,22
2,Красноярск,23
3,Волгоград,24
4,Пермь,25
5,Казань,25
6,Ростов-на-Дону,25
7,Самара,26
8,Уфа,26
9,Краснодар,30


<h2 style="color:red;">Итоги по блоку анализа работодателей.</h2>

Анализ работодателей указывает на распределение вакансий между большим количеством компаний, однако ядро формируют крупные, известные организации. Активный найм идет не только в технологическом секторе, но и крупном, корпоративном бизнесе в целом, на что указывают лидеры рынка.</br>
Количество компаний в базе шире, чем количество предложений. Не каждый работодатель участвует в активном найме.</br>
Наиболее крупные компании ведут системный и распределенный найм, охватывая значительное число городов.</br>
Большое количество компаний не имеют указанной сферы деятельности, хотя присутствуют и организации с многопрофильной моделью бизнеса.</br>
Также данные показывают о заметной доли работодателей, относящихся к сфере разработки программного обеспечения.</br>
Дополнительный анализ компании 'Яндекс' подтверждает ее широкий охват найма. Наибольшая активность наблюдается в Москве и Санкт-Петербурге, но присутствие также видно и в других крупных городах, что позволяет сделать вывод того, что крупные технологические компании строят не только столичный, но и распределенный рынок найма, сохраняя при этом основную концентрацию в крупных городах.

Подводя итог, можно сказать, что рынок работодателей неоднороден, но наиболее заметную роль играют крупные компании с широкой географией найма. IT сектор занимает значимое место.

<br><br><br><br>
# Предметный анализ

<br/><br/>
**Задание № 1**

**Количество вакансий, имеющих отношение к данным.**

In [19]:
request = f'''
select
    count(id) as data_vac_count
from vacancies
where
    name ilike '%data%' or name ilike '%данн%'
'''

task_1 = pd.read_sql_query(request, connection)

# количество вакансий, имеющих отношение к данным
display(Markdown(f"Количество вакансий, имеющих отношение к данным: **{task_1['data_vac_count'].iloc[0]}**"))

Количество вакансий, имеющих отношение к данным: **1771**

<br/><br/>
**Задание № 2**

**Количество вакансий для начинающего дата-сайентиста.**

**Требования:**

&nbsp;&nbsp;&nbsp;&nbsp;**1 - название вакансии содержит одно из сочетаний:**<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- 'data scientist'<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- 'data science'<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- 'исследователь данных'<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- 'ML, но не HTML'<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- 'machine learning'<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- 'машинн%обучен'

&nbsp;&nbsp;&nbsp;&nbsp;**2 - выполнение одного из следующих условий:**<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- в названии вакансии упоминается 'junior'<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- требуемый опыт - 'Нет опыта'<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- тип трудоустройства - 'Стажировка'

In [20]:
request = f'''
select
    count(id) as data_vac_count
from vacancies
where
    (
        name ilike '%data scientist%' 
        or name ilike '%data science%'
        or name ilike '%исследователь данных%'
        or name ilike '%ML%'
        or name ilike '%machine learning%'
        or (name ilike '%машинн%' and name ilike '%обучен%')
    )
    and name not ilike '%HTML%'
    and
    (
        name ilike '%junior%'
        or experience = 'Нет опыта'
        or employment = 'Стажировка'
    )
'''

task_2 = pd.read_sql_query(request, connection)

# количество вакансий для начинающего дата-сайентиста
display(Markdown(f"Количество вакансий для начинающего дата-сайентиста: **{task_2['data_vac_count'].iloc[0]}**"))

Количество вакансий для начинающего дата-сайентиста: **51**

<br/><br/>
**Задание № 3**

**Количество вакансий для DS, в которых в качестве ключевого навыка указан SQL или postgres.**

In [21]:
request = f'''
select
    count(id) as data_vac_count
from vacancies
where
    (
        name ilike '%data scientist%' 
        or name ilike '%data science%'
        or name ilike '%исследователь данных%'
        or name ilike '%ML%'
        or name ilike '%machine learning%'
        or (name ilike '%машинн%' and name ilike '%обучен%')
    )
    and name not ilike '%HTML%'
    and
    (
        key_skills ilike '%sql%'
        or key_skills ilike '%postgres%'
    )
'''

task_3 = pd.read_sql_query(request, connection)

# количество вакансий для DS, в которых в качестве ключевого навыка указан SQL или postgres
display(Markdown(f"Количество вакансий для DS, в которых в качестве ключевого навыка указан SQL или postgres: **{task_3['data_vac_count'].iloc[0]}**"))

Количество вакансий для DS, в которых в качестве ключевого навыка указан SQL или postgres: **229**

<br/><br/>
**Задание № 4**

**Количество вакансий для DS, в которых в качестве ключевого навыка указан Python.**

In [22]:
request = f'''
select
    count(id) as data_vac_count
from vacancies
where
    (
        name ilike '%data scientist%' 
        or name ilike '%data science%'
        or name ilike '%исследователь данных%'
        or name ilike '%ML%'
        or name ilike '%machine learning%'
        or (name ilike '%машинн%' and name ilike '%обучен%')
    )
    and name not ilike '%HTML%'
    and
    (
        key_skills ilike '%python%'
    )
'''

task_4 = pd.read_sql_query(request, connection)

# количество вакансий для DS, в которых в качестве ключевого навыка указан Python
display(Markdown(f"Количество вакансий для DS, в которых в качестве ключевого навыка указан Python: **{task_4['data_vac_count'].iloc[0]}**"))

Количество вакансий для DS, в которых в качестве ключевого навыка указан Python: **357**

<br/><br/>
**Задание № 5**

**Среднее количество ключевых навыков в вакансиях для DS (округление до двух знаков).**

In [23]:
request = f'''
select 
    round((avg((coalesce((array_length((string_to_array(key_skills,E'\t')),1)),0)))),2) as key_skills_count
from vacancies
where
    (
        name ilike '%data scientist%' 
        or name ilike '%data science%'
        or name ilike '%исследователь данных%'
        or name like '%ML%'
        or name ilike '%machine learning%'
        or (name ilike '%машинн%' and name ilike '%обучен%')
    )
    and name not ilike '%HTML%'
    and key_skills != ''
'''

task_5 = pd.read_sql_query(request, connection)

# среднее количество ключевых навыков в вакансиях для DS (округление до двух знаков)
display(Markdown(f"Среднее количество ключевых навыков в вакансиях для DS (округление до двух знаков): **{task_5['key_skills_count'].iloc[0]}**"))

Среднее количество ключевых навыков в вакансиях для DS (округление до двух знаков): **6.41**

<br/><br/>
**Задание № 6**

**Средняя зарплата дата-сайентиста по каждому варианту требования опыта (округление до целого числа).**

**Условия:**

&nbsp;&nbsp;&nbsp;&nbsp;- заполнено хотя бы одно из двух полей с заработной платой<br>
&nbsp;&nbsp;&nbsp;&nbsp;- если заполнены оба поля с заработной платой, результат считаем как сумму, деленную на двое<br>
&nbsp;&nbsp;&nbsp;&nbsp;- если заполнено одно поле, результат считаем как значение этого поля<br>

In [24]:
request = f'''
select
    experience
    ,round((avg((coalesce(salary_from, salary_to) + coalesce(salary_to, salary_from)) / 2.0)),0)::integer as avg_salary
from vacancies
where
    (
        name ilike '%data scientist%' 
        or name ilike '%data science%'
        or name ilike '%исследователь данных%'
        or name like '%ML%'
        or name ilike '%machine learning%'
        or (name ilike '%машинн%' and name ilike '%обучен%')
    )
    and name not ilike '%HTML%'
    and (salary_from is not null or salary_to is not null)
group by
    experience
'''

task_6 = pd.read_sql_query(request, connection)

# общий вид запроса
display(task_6)

# средняя зарплата дата-сайентиста по каждому варианту требования опыта (округление до целого числа)
display(Markdown(f"Средняя зарплата дата-сайентиста по каждому варианту требования опыта (округление до целого числа): **{task_6[task_6['experience']=='От 3 до 6 лет']['avg_salary'].iloc[0]}**"))

,experience,avg_salary
0,Нет опыта,74643
1,От 1 года до 3 лет,139675
2,От 3 до 6 лет,243115


Средняя зарплата дата-сайентиста по каждому варианту требования опыта (округление до целого числа): **243115**

<h2 style="color:red;">Итоги по блоку предметного анализа.</h2>

Предметный анализ показывает, что вакансии, связанные с данными, занимают заметное место в общей структуре рынка, но в целом направление остается нишевым по сравнению с более массовыми направлениями.
Малое количество вакансий для начинающих специалистов говорит о достаточно тяжелом входе в профессию. Работодатели предпочитают кандидатов с уже имеющимся опытом. Таким образом начинающим специалистам стоит ориентироваться на высокую конкурентность, стоит обращать внимание на портфолио и участие в проектах, т.к. опыт ценится.</br>
Анализ навыков указывание, что дата-сайенс является междисциплинарной областью, подразумевающей умение работать с Python, базами данных и в целом указывает на ожидание комплексной подготовки от кандидата.</br>
Данные по заработной плате показывают быстрой рост дохода в соответствии с опытом кандидата, т.е. рынок высоко оценивает наличие практического опыта у соискателей.</br>

Общий вывод показывает, что рынок дата-сайенс является перспективным, но конкурентным. Подразумевает высокий потенциал роста дохода, но и высокие требования уже на входе в профессию. Базовым языком по профессии выступает Python, также необходимо знать SQL. В остальном, широкий стек указывает на то, что дата-сайенс является комплексной деятельностью, подразумевающей навыки программирования, аналитики, машинного обучения.

<br><br><br><br>
# Дополнительный анализ данных

<br/><br/>
**Доп. аналитика - 1**

**Топ наиболее востребованных навыков в вакансиях, ориентированных на дата-сайентистов.**

In [25]:
request = f'''
-- составим общий, разобранный список требуемых навыков
with skills_draft_info as
(select
    -- приведем строки к массиву и разберем их
    unnest(string_to_array(key_skills,E'\t')) as skill
from vacancies
where
    (
        name ilike '%data scientist%' 
        or name ilike '%data science%'
        or name ilike '%исследователь данных%'
        or name like '%ML%'
        or name ilike '%machine learning%'
        or (name ilike '%машинн%' and name ilike '%обучен%')
    )
    and name not ilike '%HTML%'
    and key_skills != '')

-- составим топ-10 требуемых навыков
select
    skill
    ,count(skill) as count
    ,round(count(skill) * 100.0 / sum(count(skill)) over (), 2) as percent
from skills_draft_info
group by
    skill
order by
    count desc
limit 10

'''

add_task_1 = pd.read_sql_query(request, connection)


# выведем топ наиболее востребованных навыков в вакансиях, ориентированных на дата-сайентистов
display(Markdown(f"**Топ наиболее востребованных навыков в вакансиях, ориентированных на дата-сайентистов:**"))
display(add_task_1)

**Топ наиболее востребованных навыков в вакансиях, ориентированных на дата-сайентистов:**

,skill,count,percent
0,Python,348,12.60
1,SQL,191,6.92
2,Machine Learning,114,4.13
3,Git,66,2.39
4,Математическая статистика,62,2.25
5,Data Analysis,54,1.96
6,Pandas,52,1.88
7,Data Science,52,1.88
8,ML,49,1.77
9,Английский язык,48,1.74


<br/><br/>
**Доп. аналитика - 2**

**Топ работодателей по дата-сайенс вакансиям.**

In [26]:
request = f'''
select
   employers.name
   ,count(vacancies.id) as vac_count
   -- используем sum() over() чтобы не использовать cte для подсчета процентов
   ,round(count(vacancies.id) * 100.0 / sum(count(vacancies.id)) over (), 2) as vac_percent
from vacancies
inner join employers
    on vacancies.employer_id = employers.id
where
    (
        vacancies.name ilike '%data scientist%' 
        or vacancies.name ilike '%data science%'
        or vacancies.name ilike '%исследователь данных%'
        or vacancies.name like '%ML%'
        or vacancies.name ilike '%machine learning%'
        or (vacancies.name ilike '%машинн%' and vacancies.name ilike '%обучен%')
    )
    and vacancies.name not ilike '%HTML%'
    and key_skills != ''
group by employers.id
order by
    vac_count desc
limit 10
'''

add_task_2 = pd.read_sql_query(request, connection)

# выведем топ работодателей по дата-сайенс вакансиям
display(Markdown(f"**Топ работодателей по дата-сайенс вакансиям:**"))
display(add_task_2)

**Топ работодателей по дата-сайенс вакансиям:**

,name,vac_count,vac_percent
0,Bell Integrator,25,5.80
1,СБЕР,24,5.57
2,Банк ВТБ (ПАО),18,4.18
3,VK,15,3.48
4,Positive Technologies,11,2.55
5,Яндекс,9,2.09
6,EvenBet Gaming,9,2.09
7,МегаФон,7,1.62
8,Andersen,7,1.62
9,Бэнкс Софт Системс,6,1.39


<br/><br/>
**Доп. аналитика - 3**

**Дата-сайенс вакансии по городам.**

In [27]:
request = f'''
select
    areas.name
    ,count(vacancies.id) as vac_count
    -- используем sum() over() для подсчета процентов
    ,round(count(vacancies.id) * 100.0 / sum(count(vacancies.id)) over (), 2) as vac_percent
from vacancies
inner join areas
    on vacancies.area_id = areas.id
where
    (
        vacancies.name ilike '%data scientist%' 
        or vacancies.name ilike '%data science%'
        or vacancies.name ilike '%исследователь данных%'
        or vacancies.name like '%ML%'
        or vacancies.name ilike '%machine learning%'
        or (vacancies.name ilike '%машинн%' and vacancies.name ilike '%обучен%')
    )
    and vacancies.name not ilike '%HTML%'
    and key_skills != ''
group by areas.id
order by
    vac_count desc
limit 10
'''

add_task_3 = pd.read_sql_query(request, connection)

# выведем дата-сайенс вакансии по городам
display(Markdown(f"**Дата-сайенс вакансии по городам:**"))
display(add_task_3)

**Дата-сайенс вакансии по городам:**

,name,vac_count,vac_percent
0,Москва,188,43.62
1,Санкт-Петербург,59,13.69
2,Новосибирск,23,5.34
3,Нижний Новгород,17,3.94
4,Алматы,15,3.48
5,Казань,14,3.25
6,Томск,8,1.86
7,Минск,8,1.86
8,Армения,6,1.39
9,Краснодар,6,1.39


<br/><br/>
**Доп. аналитика - 4**

**Сравнение дата-сайенс зарплат по совокупности опыта работы и города.**

In [28]:
request = f'''
select
    areas.name
    ,experience
    ,round((avg((coalesce(salary_from, salary_to) + coalesce(salary_to, salary_from)) / 2.0)),0)::integer as avg_salary
from vacancies
inner join areas
    on vacancies.area_id = areas.id
where
    (
        vacancies.name ilike '%data scientist%' 
        or vacancies.name ilike '%data science%'
        or vacancies.name ilike '%исследователь данных%'
        or vacancies.name like '%ML%'
        or vacancies.name ilike '%machine learning%'
        or (vacancies.name ilike '%машинн%' and vacancies.name ilike '%обучен%')
    )
    and vacancies.name not ilike '%HTML%'
    and key_skills != ''
group by areas.id, experience
order by
    avg_salary desc

'''

add_task_4 = pd.read_sql_query(request, connection)

# выведем сравнение дата-сайенс зарплат по совокупности опыта работы и города
display(Markdown(f"**Сравнение дата-сайенс зарплат по совокупности опыта работы и города:**"))
display(add_task_4[add_task_4['avg_salary'].isna()==False])

**Сравнение дата-сайенс зарплат по совокупности опыта работы и города:**

,name,experience,avg_salary
64,Кипр,От 3 до 6 лет,300000.0
65,Санкт-Петербург,От 3 до 6 лет,300000.0
66,Армения,От 3 до 6 лет,268863.0
67,Москва,От 3 до 6 лет,263750.0
68,Турция,От 3 до 6 лет,233794.0
69,Черногория,От 3 до 6 лет,233794.0
70,Сербия,От 3 до 6 лет,233794.0
71,Екатеринбург,От 1 года до 3 лет,225000.0
72,Новосибирск,От 3 до 6 лет,209500.0
73,Белгород,От 3 до 6 лет,200000.0


<h2 style="color:red;">Итоги по блоку дополнительного анализа данных.</h2>
Дополнительный анализ подтверждает, что рынок дата-сайенс имеет выраженную специфику по необходимым технологическим навыкам и ориентированности на крупные города. В основном от соискателей требуется знание навыков работы с Python, SQL, анализом данных и машинным обучением.</br>
С точки зрения работодателей - спрос распределен между крупными банками, интеграторами и IT компаниями, т.е. дата-сайенс востребован в разных сферах, затрагивающих обработку данных.</br>
Географически спрос крайне неравномерен и ориентирован в основном на столицу и крупные города.

<h2 style="color:red;">Общий итог</h2>
Исследвоание показывает что рынок вакансий в сфере дата-сайенс ориентирован на крупные города (прежде всего Москва и Санкт-Петербург)</br>
Наиболее востребованными навыками являются Python, SQL, Machine Learning, умение работать с системами контроля версий, статистика и аналитика данных.</br>
Среди работодателей наиболее активны крупные компании из сфер IT, финансового и банковского блоков, сферы интеграции и операторов данных.</br>
Уровень ожидаемых доходов заметно растет с увеличением опыта.

В целом, можно уверенно сказать, что дата-сайенс является востребованной, хорошо оплачиваемой, перспективной деятельностью, подразумевающей сильную техническую подготовку и умение работать с данными.